# Crypt representation stability across a family of openings

This experiment does **not** seek an optimal anatomical boundary. It varies the opening around each saved HKS segmentation while keeping the measured parent membrane unchanged. The 20-crypt pilot is the same size-stratified selection from **20250929 and 20251201, day 3.5 onward** used in notebook 02.

We compare three methods: reparameterize each hard cut; freeze a chart on a surrounding patch but still use a hard cut; freeze that same chart and blur membership with logistic weights. Blurring weights fitting residuals, not XYZ coordinates. The middle method separates chart freezing from blurring.

The basis is fixed throughout. All references and core sampling points are frozen within each crypt/method experiment. These paired references assess boundary sensitivity; they do not replace the cross-crypt alignment reference. Outputs are validation reports and figures, not a training-data exporter. Scalar fields and latent encoders remain deferred.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/organograph").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import display
pio.renderers.default = "plotly_mimetype"
from organograph.crypt_morphology import *
from organograph.crypt_morphology.datasets import discover_hks_crypts, select_validation_crypts, load_hks_parent
from organograph.crypt_morphology.stability import METHODS
from organograph.plotting.crypt_morphology import (
    plot_opening_family, plot_boundary_stability, plot_opening_reconstructions, plot_opening_membership,
)
OUT = ROOT / "exports/crypt_morphology/boundary_stability"
OUT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get("ORGANOGRAPH_DATA_ROOT", ROOT.parent / "NicoleData"))
selected = select_validation_crypts(discover_hks_crypts(DATA_ROOT), per_group=4, seed=2026)
basis_path = ROOT / "exports/crypt_morphology/pilot_basis.json"
basis = load_basis(basis_path) if basis_path.exists() else CapBasis.build(12)
FRACTIONS = (.05, .10)
BLUR_FRACTION = .05
COLLAR_FRACTION = .30
RECOMPUTE = False  # Set True if input meshes/masks or implementation have changed.
settings = dict(basis_id=basis.identifier, max_degree=basis.max_degree,
                fractions=list(FRACTIONS), n_directions=4, include_second=True,
                max_displacement_fraction=max(FRACTIONS), collar_fraction=COLLAR_FRACTION,
                blur_fraction=BLUR_FRACTION, regularization=1e-8, selection=selected)
print(f"{len(selected)} crypts; {len(opening_shifts(1))} openings each; {basis.n_modes} vector modes")
display(pd.DataFrame(selected).groupby(["dataset", "timepoint"]).size().rename("crypts").to_frame())

## What moves, and what is measured?

Signed distance is an **edge-graph approximation** from the original rim, negative inside. Angular phase follows rim arclength; smooth harmonic extensions provide displacement fields away from the rim. The zero set of `signed_distance - displacement` defines each cut. Constant offsets expand/contract the opening. First angular modes move opposite sides in opposite directions; second modes test additional asymmetry. No detector bottom, axis, or normalized longitudinal coordinate is used.

There are 17 openings: baseline; inward/outward offsets and four sideways directions at 5% and 10% of the **original** RMS radius; four second-harmonic patterns at 10%. These amplitudes are sensitivity scales, not calibrated biological uncertainty. Cuts interpolate inside triangles, retain original geometry and barycentric provenance, and must be manifold disks. Entering a triangle protected for a neighboring crypt rejects the variant.

A fixed core consists of original triangles strictly deeper than the maximum inward displacement. We measure:

- **Core drift**: prediction changes at the same physical core points, in original coordinates, divided by the original RMS radius R₀. A rigidly aligned version is also stored.
- **Core fidelity**: reconstruction error against the actual membrane at those points. Stability alone is insufficient.
- **Whole-cap drift**: geometric RMSE on a shared cap grid after proper rigid and cap-phase alignment to each method's frozen baseline, with frozen baseline area weights. This also captures domain/extent changes and extrapolation.
- **Size sensitivity**: physical units and fixed R₀, versus normalization with the hard-cut radius recomputed for each variant. Even soft fits use that hard-cut radius, never the padded support radius.

The hard fixed-chart fit extrapolates beyond observed geometry. The soft fit weakly constrains its outer collar. A stable core is therefore not proof of a stable or anatomically valid complete output surface.

Summary medians pool accepted variants. Crypts with rejected openings contribute fewer values; these are paired sensitivity summaries, not independent biological replicates or inferential statistics.

In [ ]:
def prepare_case(record, collar_fraction=COLLAR_FRACTION):
    v, f, ids, neighbors, metadata = load_hks_parent(record)
    patch = extract_patch(v, f, ids)
    radius = surface_moments(patch.vertices, patch.faces)[2]
    family = prepare_boundary_family(v, f, ids, max_displacement=max(FRACTIONS)*radius,
                                     collar_width=collar_fraction*radius,
                                     forbidden_vertex_ids=neighbors)
    return family, radius, metadata

cache_files = [OUT / name for name in ("real_stability.csv", "diagnostics.json", "settings.json")]
cache_matches = all(p.exists() for p in cache_files) and json.loads(cache_files[-1].read_text()) == settings
if cache_matches and not RECOMPUTE:
    table = pd.read_csv(cache_files[0], dtype={"dataset": str})
    diagnostics = json.loads(cache_files[1].read_text())
    print("Loaded saved pilot results. Set RECOMPUTE=True to rerun after changing data or code.")
else:
    rows, diagnostics = [], []
    for index, record in enumerate(selected):
        identity = {k: record[k] for k in ("dataset", "timepoint", "label_uid", "crypt_index")}
        try:
            family, radius, metadata = prepare_case(record)
            result = evaluate_opening_family(family, basis, opening_shifts(radius, fractions=FRACTIONS),
                                             blur_width=BLUR_FRACTION*radius)
            rows.extend(dict(row, **identity) for row in result.rows)
            diagnostics.append(dict(result.diagnostics, **metadata))
        except ValueError as error:
            rows.extend(dict(identity, variant=s.name, method=m, status="failed_setup", error=str(error))
                        for s in opening_shifts(1, fractions=FRACTIONS) for m in METHODS)
            diagnostics.append(dict(identity, error=str(error)))
        print(f"{index+1}/{len(selected)}: {record['label_uid']}")
    table = pd.DataFrame(rows)
    table.to_csv(cache_files[0], index=False)
    cache_files[1].write_text(json.dumps(diagnostics, indent=2))
    cache_files[2].write_text(json.dumps(settings, indent=2))
expected = len(selected)*len(opening_shifts(1, fractions=FRACTIONS))*len(METHODS)
assert len(table) == expected, "Missing requested variants; inspect the run before summarizing"
display(table.groupby(["method", "status"]).size().rename("fits").to_frame())
failures = table[table.status != "ok"]
print(f"Failures/rejections: {len(failures)} of {len(table)} requested fits")
if len(failures):
    display(failures[["dataset", "label_uid", "crypt_index", "variant", "method", "status", "error"]])
success = table[(table.status == "ok") & (table.variant != "baseline")].copy()
if success.empty:
    raise RuntimeError("No valid nonbaseline results; inspect failures")
metrics = ["core_drift_relative", "core_fidelity_relative", "cap_drift_fixed_rms", "cap_drift_unit_rms"]
print("Relative RMSE summaries (fractions, not percentages); accepted nonbaseline variants only:")
display(success.groupby("method")[metrics].agg(["median", "max"]))

In [ ]:
figure = plot_boundary_stability(table)
figure.savefig(OUT / "real_stability.png", dpi=170)
display(figure)
plt.close(figure)
print("Conditioning and rank (including baselines):")
display(table[table.status == "ok"].groupby("method")[["condition_number", "data_rank"]].agg(["min", "median", "max"]))
print("Fixed-core fraction and mesh resolution, one row per crypt:")
diag = pd.DataFrame(diagnostics)
display(diag[["dataset", "label_uid", "crypt_index", "core_area_fraction", "median_edge_length",
              "max_displacement_in_median_edges", "protected_support_area_fraction"]])
print("Change in measured hard-cut size and area (one row per variant):")
display(success[success.method == "recut_hard"][["rms_fraction", "area_fraction"]].describe())

## Inspect an opening family and its reconstructed surfaces

By default, select the crypt with the largest **recut core drift**. Change `INSPECT_KEY` to inspect another `(dataset, label_uid, crypt_index)`. Black is the original opening; colored curves are prescribed alternatives on the unchanged membrane. The four selected first-mode phases test both directions along two arbitrary rim axes; they are not anatomical orientation labels.

In [ ]:
INSPECT_KEY = None
worst = success[success.method == "recut_hard"].sort_values("core_drift_relative").iloc[-1]
key = INSPECT_KEY or (worst.dataset, worst.label_uid, int(worst.crypt_index))
record = next(r for r in selected if (r["dataset"], r["label_uid"], r["crypt_index"]) == key)
family, radius, metadata = prepare_case(record)
shifts = opening_shifts(radius, fractions=FRACTIONS)
inspection = evaluate_opening_family(family, basis, shifts, blur_width=BLUR_FRACTION*radius, store_samples=True)
print("Inspecting", key)
print("RMS radius in source units:", radius)
print("Graph-distance approximation; maximum shift / median edge:", family.diagnostics["max_displacement_in_median_edges"])
show_shifts = [s for s in shifts if s.name == "baseline" or s.name.startswith("offset_") or s.name.startswith("tilt_0.100")]
fig = plot_opening_family(family, show_shifts, support=inspection.support)
fig.write_html(OUT / "opening_family.html", include_plotlyjs=True)
display(fig)
fig = plot_opening_membership(family, inspection.support, shifts[0], blur_width=BLUR_FRACTION*radius)
fig.write_html(OUT / "soft_membership.html", include_plotlyjs=True)
display(fig)

The next figure compares full-domain reconstructions in the original physical coordinates. Blue is reconstructed geometry; gray is the fitted membrane (the whole padded support for the soft method). All panels use the same spatial limits. Fixed-chart surfaces extend to the computational support boundary: their outer regions are extrapolated or weakly constrained. Do not interpret that rim as a newly discovered anatomical opening.

In [ ]:
case_table = pd.DataFrame(inspection.rows)
valid_case = case_table[(case_table.status == "ok") & (case_table.variant != "baseline")]
variant = valid_case[valid_case.method == "recut_hard"].sort_values("core_drift_relative").iloc[-1].variant
fig = plot_opening_reconstructions(inspection, basis, variant)
fig.write_html(OUT / "opening_reconstructions.html", include_plotlyjs=True)
display(fig)
display(case_table[case_table.variant.isin(["baseline", variant])][
    ["variant", "method", "status", "core_fidelity_relative", "core_drift_relative",
     "cap_drift_fixed_rms", "cap_drift_unit_rms", "condition_number"]])
# Also expose the most unstable outer extrapolation, which may be a different shift.
cap_variant = valid_case[valid_case.method == "fixed_chart_hard"].sort_values("cap_drift_fixed_rms").iloc[-1].variant
if cap_variant != variant:
    print("Largest fixed-chart whole-cap drift for this crypt:", cap_variant)
    fig = plot_opening_reconstructions(inspection, basis, cap_variant)
    fig.write_html(OUT / "worst_outer_extrapolation.html", include_plotlyjs=True)
    display(fig)
    display(case_table[case_table.variant == cap_variant][["variant", "method", "core_fidelity_relative", "core_drift_relative", "cap_drift_fixed_rms", "condition_number"]])

## Synthetic mesh-resolution control

Use the same analytic asymmetric surface and the same opening at 75% of its parameter range, with three mesh resolutions. That parameter is used only to construct a known synthetic input mask; the representation/perturbation code receives geometry and membership alone. The reference basis, fractional displacement, blur width, and support width remain fixed. Compare metrics as well as geometric resolution; graph distance can retain directional bias even when edge lengths decrease. A separate analytic planar-disk unit test checks signed distance and clipped area under refinement.

In [ ]:
synthetic_rows = []
for rings, sectors in [(16, 32), (24, 48), (32, 64)]:
    v, f = synthetic_crypt("asymmetric", n_rings=rings, n_sectors=sectors)
    ids = np.arange(1 + (3*rings//4)*sectors)
    patch = extract_patch(v, f, ids)
    radius_s = surface_moments(patch.vertices, patch.faces)[2]
    family_s = prepare_boundary_family(v, f, ids, max_displacement=.1*radius_s, collar_width=.3*radius_s)
    result_s = evaluate_opening_family(family_s, basis, opening_shifts(radius_s), blur_width=.05*radius_s)
    synthetic_rows.extend(dict(row, rings=rings, sectors=sectors) for row in result_s.rows)
synthetic = pd.DataFrame(synthetic_rows)
synthetic.to_csv(OUT / "synthetic_resolution.csv", index=False)
display(synthetic.groupby(["rings", "method", "status"]).size().rename("fits").to_frame())
display(synthetic[(synthetic.status == "ok") & (synthetic.variant != "baseline")]
        .groupby(["rings", "method"])[metrics].median())

## Blur-width and surrounding-support controls

On the inspected real crypt, compare blur widths 2.5% and 5% of R₀, with support collars 30% and 40%. The maximum boundary shift stays at 10%. This small sweep is diagnostic, not a parameter optimization. Each support gets its own frozen baseline; therefore cap-drift values compare **within-support** variation, not absolute agreement between supports. Core fidelity and core drift are directly comparable because the physical core is identical.

The main pilot used the 30% support and 5% blur. The collar terminates four blur widths beyond the furthest permitted outward offset in that setting. Increasing the collar tests finite-support effects, including effects of changing the chart. Neighbor-protected faces remain zero-weighted.

The support is changed with `family.with_collar_width`, keeping the distance and angular displacement fields exactly fixed. Recomputing those fields on each wider support would also change the openings and confound this control.

In [ ]:
sweep_rows = []
sweep_shifts = [s for s in shifts if s.name in ["baseline", "offset_-0.100", "offset_+0.100", "tilt_0.100_0", "tilt_0.100_1"]]
for collar in [.30, .40]:
    for blur in [.025, .05]:
        try:
            fam, rad = family.with_collar_width(collar*radius), radius
            result = evaluate_opening_family(fam, basis, sweep_shifts, blur_width=blur*rad)
            sweep_rows.extend(dict(row, collar_fraction=collar, blur_fraction=blur) for row in result.rows)
        except ValueError as error:
            sweep_rows.extend(dict(variant=s.name, method=m, status="failed_setup", error=str(error),
                                   collar_fraction=collar, blur_fraction=blur) for s in sweep_shifts for m in METHODS)
sweep = pd.DataFrame(sweep_rows)
sweep.to_csv(OUT / "blur_support_sweep.csv", index=False)
display(sweep.groupby(["collar_fraction", "blur_fraction", "method", "status"]).size().rename("fits").to_frame())
if (sweep.status != "ok").any():
    display(sweep[sweep.status != "ok"])
display(sweep[(sweep.status == "ok") & (sweep.variant != "baseline")]
        .groupby(["collar_fraction", "blur_fraction", "method"])[metrics + ["condition_number"]].median())
# The unweighted recut control must be unchanged by either soft-fit setting.
recut_control = sweep[sweep.method == "recut_hard"]
for name, values in recut_control.groupby("variant"):
    assert values.status.eq("ok").all()
    assert np.ptp(values.core_drift_relative) < 1e-12
print("Control passed: identical recut surfaces/results across blur and support settings.")

## Reading the results before choosing a representation

Compare fixed-chart hard vs recut hard to assess chart changes, then fixed-chart soft vs fixed-chart hard to assess blurring. Compare core drift **and** fidelity; inspect the full-cap surface and conditioning before deciding whether a stable core supports using all coefficients downstream. Compare fixed-R₀ and recomputed-RMS errors to identify sensitivity introduced by normalization.

These local perturbations do not establish independence from every possible initial detection, preserve a unique anatomical correspondence, or guarantee that arbitrary coefficient vectors decode to valid crypts. No method is selected automatically from these results. Review the pilot and controls before proceeding to scalar fields or batch export. All numerical reports and interactive HTML figures are saved under `exports/crypt_morphology/boundary_stability/`.